<!-- COMMONS LAUNCHER v4 · generated by tools/notebooks.py · do not edit by hand -->
<a href="https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials"><img src="https://raw.githubusercontent.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master/brand/synapsa-commons-badge.png" alt="Synapsa Commons" height="36"></a>

Free, hands-on AI courses that run anywhere, from the team building [Synapsa](https://synapsa.realai.eu), an AI-native
learning platform.

© 2026 RealAI · free to learn from, share and adapt, not to sell ([CC BY-NC-SA 4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/)).
The notice at the end of this notebook says what you may and may not do.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/blob/master/lessons/T00-L03-one-client-three-backends/lesson.ipynb)
[![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/blob/master/lessons/T00-L03-one-client-three-backends/lesson.ipynb)
[![Open in Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master?labpath=lessons/T00-L03-one-client-three-backends/lesson.ipynb)
[![Open in Codespaces](https://github.com/codespaces/badge.svg)](https://codespaces.new/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials)

This lesson needs Python 3.11 or newer.

In [ ]:
# --- COMMONS LAUNCHER v4 · generated by tools/notebooks.py · do not edit by hand ---
# Makes this notebook run anywhere. Every line is a no-op when the thing is already present,
# so a local clone pays nothing and an online notebook repairs itself.
import importlib.util, os, subprocess, sys, urllib.request
from pathlib import Path

COMMONS_PIP = []            # (import name, pinned pip spec) for what this lesson imports
COMMONS_SIBLINGS = []    # files that must sit beside the notebook
# A fork, a classroom mirror or an offline copy can serve the files from elsewhere by setting
# COMMONS_RAW_OVERRIDE before running this cell.
COMMONS_RAW = os.environ.get("COMMONS_RAW_OVERRIDE") or "https://raw.githubusercontent.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master/lessons/T00-L03-one-client-three-backends/"

# Resolve siblings against the LESSON's own directory, not the working directory. A notebook
# has no __file__ and runs with cwd alongside itself; a grader imports this file from the repo
# root. Checking cwd blindly makes the grader think every sibling is missing and reach for the
# network -- which would put a download on a graded path.
try:
    COMMONS_DIR = Path(__file__).resolve().parent
except NameError:
    COMMONS_DIR = Path.cwd()


def commons_host() -> str:
    """Name the notebook service we are on. Used for the message, and for honest errors."""
    try:
        if importlib.util.find_spec("google.colab") is not None:
            return "Google Colab"
    except (ImportError, ValueError):
        pass
    if os.environ.get("KAGGLE_KERNEL_RUN_TYPE"):
        return "Kaggle"
    if os.environ.get("BINDER_SERVICE_HOST"):
        return "Binder"
    if os.environ.get("CODESPACES"):
        return "GitHub Codespaces"
    return "a local Python environment"


_missing = [pip for imp, pip in COMMONS_PIP if importlib.util.find_spec(imp) is None]
if _missing:
    print("installing " + ", ".join(_missing) + " ...")
    # pip everywhere a student is likely to be; uv-managed local venvs ship without pip.
    if importlib.util.find_spec("pip") is not None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *_missing], check=True)
    else:
        subprocess.run(["uv", "pip", "install", "-q", "--python", sys.executable, *_missing],
                       check=True)
    importlib.invalidate_caches()

_fetched = []
for _name in COMMONS_SIBLINGS:
    if not (COMMONS_DIR / _name).exists():
        (COMMONS_DIR / _name).parent.mkdir(parents=True, exist_ok=True)
        try:
            urllib.request.urlretrieve(COMMONS_RAW + _name, COMMONS_DIR / _name)
            _fetched.append(_name)
        except Exception as _e:  # Kaggle disables the internet by default; say so plainly
            raise RuntimeError(
                f"this lesson needs {_name} beside the notebook and could not fetch it "
                f"({_e}). On Kaggle, switch Internet on in the notebook settings panel "
                f"(Kaggle allows that only for phone-verified accounts); otherwise download it "
                f"from {COMMONS_RAW + _name} and upload it beside the notebook."
            ) from None

print("ready on " + commons_host() + ("; fetched " + ", ".join(_fetched) if _fetched else ""))
# --- END COMMONS LAUNCHER ---

# T00-L03 · One client, three backends: the normalising half of an LLM client, and the suite that tests it (run against three hand-written stand-in backends)

**You will build:** the layer of an LLM client that turns three different backends' answers into
one internal type. It reads tool calls, reasoning and JSON out of whatever shape a backend
sends them in. Then you will write the compatibility suite that reports, for every backend and
every feature, whether the feature arrived as documented, arrived under another name, was
silently dropped, or was rejected outright.

**Time:** ~50 minutes · **Runs on:** a laptop CPU, 8 GiB RAM, no GPU, no network, no API key
· **Prerequisites:** `T00-L01-the-8gb-track` (only for the `cpu8` tier vocabulary; nothing from
it is imported). Python dictionaries, JSON and regular expressions.

By the end you will be able to:

1. Implement `parse_tool_calls`, which reads a tool call whose arguments arrive as a JSON
   string, an object, an empty string or a legacy `function_call`, and refuses a malformed
   call with an error instead of running it.
2. Implement `split_reasoning`, which recovers a model's reasoning from four different places
   and removes it from the visible answer.
3. Implement `recover_json`, which finds the JSON object in an answer that was meant to be
   pure JSON but arrived wrapped in prose or a code fence.
4. Implement `normalise`, which composes those three into one `Turn` and maps errors, finish
   reasons and token counts onto one vocabulary.
5. Classify every (backend, feature) cell of a compatibility suite as kept, renamed, dropped
   or rejected, and explain why an HTTP 200 does not prove a request field was honoured.

**What is real here and what is not.** This lesson runs with no network and no API key, so the
three "backends" are **hand-written replays**, not recordings of live traffic. Each of the
twelve responses was written by hand to the shape that a product's own documentation describes
on the snapshot date printed below; none was captured from a live server. Every identifier,
model name, token count and error message in them is a placeholder, and the error wording is
invented. **Nothing in this notebook shows what any provider returns today.** What is real is
the client code you write, and the field names and behaviours that `claims.yaml` sources to
each provider's documentation on the snapshot date. The lesson teaches the method; to learn a
provider's behaviour today you re-record the probes against it and update the snapshot date.

<!-- COMMONS BEFORE v1 · generated from meta.yaml by tools/notebooks.py · do not edit by hand -->
## Before you start

**Lessons to do first:** `T00-L01`

**This lesson assumes you can already:**

- write a Python function that walks nested dicts and lists, such as the JSON a web API returns
- parse JSON with the json module and handle a JSONDecodeError
- write a regular expression that matches the text between two tags, and say what greedy and non-greedy matching do
- say what an HTTP 200 and a 4xx status mean, and what a chat request body looks like (a list of role and content messages)

**New to some of that? These are good places to learn it** (free to read online first, then paid books). You do not need any of them if the list above already reads as familiar.

- [Regular Expression HOWTO](https://docs.python.org/3/howto/regex.html) · Python Software Foundation · *free* — greedy versus non-greedy matching and re.DOTALL, the two things that decide whether a think block is removed cleanly in split_reasoning
- [json: JSON encoder and decoder](https://docs.python.org/3/library/json.html) · Python Software Foundation · *free* — json.loads, JSONDecodeError and JSONDecoder.raw_decode, the building blocks for reading tool arguments and for recover_json
- [HTTP response status codes](https://developer.mozilla.org/en-US/docs/Web/HTTP/Reference/Status) · MDN Web Docs · *free* — the 2xx and 4xx families, which is the vocabulary behind sorting error bodies and the point that a 200 does not mean a field was honoured
- [OpenAI SDK compatibility (Claude Platform docs)](https://platform.claude.com/docs/en/cli-sdks-libraries/libraries/openai-sdk) · Anthropic · *free* — a real backend's table of OpenAI request fields it fully supports, caps, restricts or silently ignores, the kept-or-dropped question the compatibility suite asks
- [LLaMA.cpp HTTP Server](https://github.com/ggml-org/llama.cpp/tree/master/tools/server) · ggml-org / llama.cpp project · *free* — the README of a local OpenAI-compatible server, including its reasoning_content and --reasoning-format options that the local stand-in backend imitates

In [ ]:
# Setup: everything the lesson needs, in one cell. Standard library only.
import contextlib
import copy
import io
import json
import os
import re
import signal
import sys
import threading
import time
import traceback
from dataclasses import dataclass
from typing import Any, Callable

print("python", sys.version.split()[0])

# The day the field names in section 1's replays were checked against the documentation of
# each product they imitate. Provider field names change more often than any algorithm does, so this date is
# part of every result the lesson prints.
SNAPSHOT_DATE = "2026-09-29"

_FAILED_CHECKS: list[str] = []

# The exercises, in the order you meet them, and the functions each one asks you to write.
# The progress board at the foot of the notebook is built from this, and a cell that is
# waiting on an unfinished exercise names it from here.
_EXERCISES: dict[str, tuple[str, ...]] = {
    "exercise 1": ("parse_tool_calls",),
    "exercise 2": ("split_reasoning",),
    "exercise 3": ("recover_json",),
    "exercise 4": ("normalise",),
    "exercise 5": ("classify",),
}
_STATUS: dict[str, str] = {}   # label -> "passed" | "failed" | "not started", latest run


def _named(labels: list[str]) -> str:
    """["exercise 3"] -> "exercise 3 (recover_json)"; several -> "exercises 3, 4 and 5"."""
    if len(labels) == 1:
        return f"{labels[0]} ({', '.join(_EXERCISES[labels[0]])})"
    nums = [label.split()[-1] for label in labels]
    return "exercises " + ", ".join(nums[:-1]) + " and " + nums[-1]


# How long one check may run before it counts as stuck. A right answer takes milliseconds.
_CHECK_SECONDS = 5


class _TooSlow(BaseException):
    """Raised by `_time_limit`. It derives from BaseException, not Exception, so a broad
    `except Exception:` inside a loop under test cannot swallow it and keep the loop spinning."""


@contextlib.contextmanager
def _time_limit(seconds: float, what: str):
    """Turn a call that never returns into a failed check that says so, instead of a notebook
    that hangs. It uses the operating system's alarm where there is one (Linux and macOS, in the
    main thread) and does nothing elsewhere. The alarm repeats every quarter of a second until
    the call gives up, so even a bare `except:` in the loop cannot hold it for long. An alarm that
    whoever started us had already set (the grader's own time limit) keeps its deadline if it is
    due first, and is put back, with the time it had left, when the check ends. A right answer
    finishes far inside the limit, and nothing that is printed depends on it."""
    if not (all(hasattr(signal, name) for name in ("setitimer", "SIGALRM", "pthread_sigmask", "sigpending", "sigwait"))
            and threading.current_thread() is threading.main_thread()):
        yield
        return
    message = (f"{what} was still running after {seconds:g} s: a loop that does not "
               "move forward on some input, such as a tag or a brace that never "
               "closes, never ends")
    started = time.monotonic()
    previous = signal.getsignal(signal.SIGALRM)
    outer_left, outer_every = signal.getitimer(signal.ITIMER_REAL)   # an alarm already running
    # One alarm at a time, never a repeating one. next: when ours is due; outer: when the alarm we
    # found is due (every: its interval); unseen: an alarm is armed and has not reached _fire yet.
    state = {"next": started + seconds, "outer": started + outer_left if outer_left > 0 else None,
             "every": outer_every, "unseen": False, "fired": False, "done": False,
             "me": sys._getframe(), "owner": None}
    owner = state["me"].f_back
    while owner is not None and owner.f_globals is vars(contextlib):    # contextlib's __enter__
        owner = owner.f_back
    state["owner"] = owner                               # the frame running the `with`

    def _arm(now, soonest=0.001):                        # one shot, at whichever deadline is next
        due = [t for t in (state["next"], state["outer"]) if t is not None]
        if due:
            signal.setitimer(signal.ITIMER_REAL, max(min(due) - now, soonest))
            state["unseen"] = True

    def _in_guard(frame):                                # is the alarm landing in this guard's own
        while frame is not None and frame is not state["owner"]:     # set-up, exit or clean-up (or
            if (frame is state["me"] or frame.f_code in (_fire.__code__, _arm.__code__)    # in code
                    or (frame.f_back is state["owner"] and frame.f_globals is vars(contextlib))):
                return True                              # run from contextlib's __exit__ under the `with`)?
            frame = frame.f_back
        return False

    def _fire(signum, frame):
        state["unseen"] = False
        if state["done"]:
            return                                       # a late alarm of ours: already handled
        if _in_guard(frame):                             # never raise inside the guard: it is
            _arm(time.monotonic(), 0.01)                 # finishing; look again shortly
            return
        now = time.monotonic()
        if state["outer"] is not None and now >= state["outer"] - 0.005:
            every = state["every"]                       # the outer alarm is due: act as it would
            state["outer"] = state["outer"] + every * (1 + (now - state["outer"]) // every) if every else None
            try:
                if callable(previous):
                    previous(signum, frame)
                elif previous != signal.SIG_IGN:         # its default action ends the process
                    signal.signal(signal.SIGALRM, signal.SIG_DFL)
                    os.kill(os.getpid(), signal.SIGALRM)
            except BaseException:
                state["next"] = None                     # the outer one is stopping the block
                raise
            finally:
                left, every = signal.getitimer(signal.ITIMER_REAL)
                if left > 0:                             # the outer handler armed its next alarm
                    state["outer"], state["every"] = time.monotonic() + left, every
                _arm(time.monotonic())
            now = time.monotonic()
        if state["next"] is not None and now >= state["next"] - 0.005:
            state["fired"] = True
            state["next"] = now + 0.25                   # again, in case a bare `except:` swallows it
            _arm(now)
            raise _TooSlow(message, state)
        _arm(now)

    signal.signal(signal.SIGALRM, _fire)
    _arm(time.monotonic())
    try:
        try:
            yield
        finally:
            state["done"] = True                         # from here on an alarm of ours only counts
            mask = signal.pthread_sigmask(signal.SIG_BLOCK, {signal.SIGALRM})
            try:
                if signal.setitimer(signal.ITIMER_REAL, 0)[0] == 0 and state["unseen"]:
                    # Ours went off and has not reached _fire: it is pending here, or on its way to
                    # another thread. Take it before the old handler (by default, one that ends the
                    # process) is back.
                    give_up = time.monotonic() + 0.5
                    while state["unseen"] and time.monotonic() < give_up:
                        if signal.SIGALRM in signal.sigpending():
                            signal.sigwait({signal.SIGALRM})
                            state["unseen"] = False
                        else:
                            time.sleep(0.001)            # another thread has it: _fire runs next
                signal.signal(signal.SIGALRM, signal.SIG_DFL if previous is None else previous)
                outer = state["outer"]                   # re-arm the alarm we found, last
                signal.setitimer(signal.ITIMER_REAL, *((max(outer - time.monotonic(), 0.001), state["every"])
                                                       if outer is not None else (0, 0)))
            finally:
                signal.pthread_sigmask(signal.SIG_SETMASK, mask)
                state["me"] = state["owner"] = None
    except _TooSlow as exc:
        if exc.args[-1] is not state:                    # an enclosing guard's: not ours to report
            raise
        raise _TooSlow(message) from None
    if state["fired"]:                                   # the block caught our alarm and carried on
        raise _TooSlow(message)


def _try(label: str, check: Callable[[], None], needs: tuple[str, ...] = ()) -> None:
    """Run a check, or a demo that depends on your code, without derailing the notebook.

    A stub you have not filled in yet simply says so. A wrong answer prints the check's own
    message, which names the likely mistake, and the notebook carries on, so one broken
    exercise never hides the feedback on the others. A demo names the exercises it `needs`:
    until each has passed its check, the demo says which one it is waiting for and skips. A
    check still running after `_CHECK_SECONDS` is stopped and counts as failed, so a loop that
    never ends cannot hang the notebook. Nothing is swallowed: every outcome is recorded in `_STATUS` for the progress board at the
    foot of the notebook, and every failure in `_FAILED_CHECKS`, which ends a script run
    non-zero.
    """
    waiting = [name for name in _EXERCISES   # in the order you meet them
               if name in needs and _STATUS.get(name) != "passed"]
    if waiting:
        _STATUS[label] = "not started"
        print(f"{label}: skipped, needs {_named(waiting)} to pass first.")
        return
    try:
        with _time_limit(_CHECK_SECONDS, "this check"):
            check()
    except _TooSlow as exc:
        _STATUS[label] = "failed"
        _FAILED_CHECKS.append(label)
        print(f"{label}: FAILED, {exc}")
    except NotImplementedError as exc:
        _STATUS[label] = "not started"
        stub = traceback.extract_tb(exc.__traceback__)[-1].name   # the frame that raised
        owner = [name for name, funcs in _EXERCISES.items() if stub in funcs and name != label]
        if owner:
            print(f"{label}: skipped, needs {_named(owner)} first.")
        elif label in _EXERCISES:
            print(f"{label}: not implemented yet, fill in {stub}() above, then re-run "
                  "this cell.")
        else:
            print(f"{label}: skipped, {stub}() is not implemented yet.")
    except AssertionError as exc:
        _STATUS[label] = "failed"
        _FAILED_CHECKS.append(label)
        print(f"{label}: FAILED, {exc}")
    except Exception as exc:  # a half-finished implementation raising something else
        _STATUS[label] = "failed"
        _FAILED_CHECKS.append(label)
        print(f"{label}: raised {type(exc).__name__}: {exc}")
    else:
        _STATUS[label] = "passed"

## 1. The replays: three hand-written stand-ins

Three stand-ins, chosen because they are the three places a learner actually points a client:

- **`local`**: a server on your own machine that speaks the OpenAI chat shape, in the style of
  `llama-server` from llama.cpp.
- **`hosted`**: an aggregator's free tier, in the style of OpenRouter, where one URL fronts
  many models.
- **`frontier`**: a paid frontier model reached through its vendor's OpenAI-compatibility
  layer, in the style of Anthropic's.

None of them is the product it is styled after: each is a hand-written imitation of what that
product's documentation describes, so it shows a *documented* behaviour, never a measured one.
The code calls them `RECORDED` because a client replays them, but nobody recorded them.

Each is asked the same four questions, called **probes**: a plain greeting (`plain`), a
question with a weather tool attached (`tools`), a question with `reasoning_effort` set
(`reasoning`), and an extraction request with a JSON schema in `response_format` (`schema`).
Each replayed answer is stored as the pair `(HTTP status, JSON body)`, and the cell below
counts them. Read them. The differences you are about to normalise are all in there.

In [ ]:
# The replays. Written BY HAND to each backend's documented shape (see claims.yaml), not
# captured from any server, so a reader can see every field. Every id, model name, token count
# and error message here is a made-up placeholder.
RECORDED: dict[str, dict[str, tuple[int, dict]]] = {
    "local": {
        "plain": (200, {
            "id": "chatcmpl-local-1", "object": "chat.completion", "created": 1790000000,
            "model": "local-instruct-q4",
            "choices": [{"index": 0, "finish_reason": "stop",
                         "message": {"role": "assistant",
                                     "content": "Hello. How can I help you today?"}}],
            "usage": {"prompt_tokens": 14, "completion_tokens": 9, "total_tokens": 23}}),
        "tools": (200, {
            "id": "chatcmpl-local-2", "object": "chat.completion", "created": 1790000001,
            "model": "local-instruct-q4",
            "choices": [{"index": 0, "finish_reason": "tool_calls",
                         "message": {"role": "assistant", "content": "",
                                     "tool_calls": [{"type": "function", "id": "call_Kq3vT9",
                                                     "function": {"name": "get_weather",
                                                                  "arguments": "{\"city\":\"Lisbon\"}"}}]}}],
            "usage": {"prompt_tokens": 96, "completion_tokens": 21, "total_tokens": 117}}),
        "reasoning": (200, {
            "id": "chatcmpl-local-3", "object": "chat.completion", "created": 1790000002,
            "model": "local-instruct-q4",
            "choices": [{"index": 0, "finish_reason": "stop",
                         "message": {"role": "assistant",
                                     "content": "<think>\nBoth start with 9. Then 0.9 is nine tenths and 0.11 is only "
                                                "a little over one tenth, so 9.9 is larger.\n</think>\n\n"
                                                "9.9 is larger than 9.11.",
                                     "reasoning_content": "Both start with 9. Then 0.9 is nine tenths and 0.11 is only "
                                                          "a little over one tenth, so 9.9 is larger."}}],
            "usage": {"prompt_tokens": 27, "completion_tokens": 58, "total_tokens": 85}}),
        "schema": (200, {
            "id": "chatcmpl-local-4", "object": "chat.completion", "created": 1790000003,
            "model": "local-instruct-q4",
            "choices": [{"index": 0, "finish_reason": "stop",
                         "message": {"role": "assistant",
                                     "content": "{\"city\":\"Lisbon\",\"temp_c\":21}"}}],
            "usage": {"prompt_tokens": 61, "completion_tokens": 14, "total_tokens": 75}}),
    },
    "hosted": {
        "plain": (200, {
            "id": "gen-hosted-1", "object": "chat.completion", "created": 1790000010,
            "model": "hosted/free-instruct:free",
            "choices": [{"index": 0, "finish_reason": "stop",
                         "message": {"role": "assistant", "content": "Hi there! What would you like to do?",
                                     "refusal": None, "reasoning": None}}],
            "usage": {"prompt_tokens": 12, "completion_tokens": 10, "total_tokens": 22}}),
        "tools": (200, {
            "id": "gen-hosted-2", "object": "chat.completion", "created": 1790000011,
            "model": "hosted/free-instruct:free",
            "choices": [{"index": 0, "finish_reason": "tool_calls",
                         "message": {"role": "assistant", "content": None, "refusal": None,
                                     "reasoning": None,
                                     "tool_calls": [{"type": "function", "id": "call_9fA2c1", "index": 0,
                                                     "function": {"name": "get_weather",
                                                                  "arguments": "{\n  \"city\": \"Lisbon\"\n}"}}]}}],
            "usage": {"prompt_tokens": 91, "completion_tokens": 19, "total_tokens": 110}}),
        "reasoning": (200, {
            "id": "gen-hosted-3", "object": "chat.completion", "created": 1790000012,
            "model": "hosted/free-instruct:free",
            "choices": [{"index": 0, "finish_reason": "stop",
                         "message": {"role": "assistant", "content": "9.9 is larger than 9.11.",
                                     "refusal": None,
                                     "reasoning": "Compare the tenths: 9 against 1. Nine is more, so 9.9 wins.",
                                     "reasoning_details": [
                                         {"type": "reasoning.text", "index": 0, "format": "unknown",
                                          "text": "Compare the tenths: 9 against 1. Nine is more, so 9.9 wins."}]}}],
            "usage": {"prompt_tokens": 27, "completion_tokens": 44, "total_tokens": 71,
                      "completion_tokens_details": {"reasoning_tokens": 30}}}),
        "schema": (400, {
            "error": {"code": 400, "metadata": {},
                      "message": "[invented wording] This model does not support structured outputs "
                                 "(response_format of type json_schema)."}}),
    },
    "frontier": {
        "plain": (200, {
            "id": "msg-frontier-1", "object": "chat.completion", "created": 1790000020,
            "model": "frontier-model",
            "choices": [{"index": 0, "finish_reason": "stop",
                         "message": {"role": "assistant", "content": "Hello! How can I help?"}}],
            "usage": {"prompt_tokens": 9, "completion_tokens": 8, "total_tokens": 17}}),
        "tools": (200, {
            "id": "msg-frontier-2", "object": "chat.completion", "created": 1790000021,
            "model": "frontier-model",
            "choices": [{"index": 0, "finish_reason": "tool_calls",
                         "message": {"role": "assistant", "content": "I will check the weather in Lisbon.",
                                     "tool_calls": [{"type": "function", "id": "call_f2c9d4",
                                                     "function": {"name": "get_weather",
                                                                  "arguments": "{\"city\": \"Lisbon\"}"}}]}}],
            "usage": {"prompt_tokens": 112, "completion_tokens": 33, "total_tokens": 145}}),
        "reasoning": (200, {
            "id": "msg-frontier-3", "object": "chat.completion", "created": 1790000022,
            "model": "frontier-model",
            "choices": [{"index": 0, "finish_reason": "stop",
                         "message": {"role": "assistant",
                                     "content": "9.9 is larger than 9.11, because 0.90 is greater than 0.11."}}],
            "usage": {"prompt_tokens": 24, "completion_tokens": 21, "total_tokens": 45}}),
        "schema": (200, {
            "id": "msg-frontier-4", "object": "chat.completion", "created": 1790000023,
            "model": "frontier-model",
            "choices": [{"index": 0, "finish_reason": "stop",
                         "message": {"role": "assistant",
                                     "content": "Here is the extracted data:\n\n```json\n{\n  \"city\": \"Lisbon\",\n"
                                                "  \"temp_c\": 21\n}\n```"}}],
            "usage": {"prompt_tokens": 58, "completion_tokens": 30, "total_tokens": 88}}),
    },
}
BACKENDS = tuple(RECORDED)
PROBES = ("plain", "tools", "reasoning", "schema")
print(f"as of {SNAPSHOT_DATE}: {len(RECORDED)} backends x {len(PROBES)} probes = "
      f"{sum(len(v) for v in RECORDED.values())} replayed answers")
print("  hand-written to each product's documented shape; not captured from any live server")
for _backend in BACKENDS:
    _codes = [RECORDED[_backend][p][0] for p in PROBES]
    print(f"  {_backend:<9} HTTP status per probe {dict(zip(PROBES, _codes))}")

Here is the same tool call from all three stand-ins, side by side, exactly as the replays hold it.
The model asked for the same thing, `get_weather` for Lisbon. Look at the `arguments` field:
it is not an object, it is a **string that contains JSON**, and each stand-in formats that
string differently. Then look at the `reasoning` probe's messages: where does each stand-in put
what the model was thinking?

In [ ]:
for _backend in BACKENDS:
    _msg = RECORDED[_backend]["tools"][1]["choices"][0]["message"]
    print(f"{_backend:<9} tool call  : {json.dumps(_msg['tool_calls'][0]['function']['arguments'])}")
print()
for _backend in BACKENDS:
    _status, _body = RECORDED[_backend]["reasoning"]
    _msg = _body["choices"][0]["message"]
    print(f"{_backend:<9} reasoning message keys: {sorted(_msg)}")
print("\nThe local answer also carries `<think>` tags inside `content`: that stand-in is written as a")
print("server run in a mode that keeps the tags AND fills a separate field. Exercise 2 has to survive that.")

## 2. The client around your code

Everything in the next cell is given. `build_request` makes the four probes in the OpenAI
chat shape, `Replay` plays the part of one backend's HTTP endpoint, and `Client.complete`
is the whole client: send a request, hand the raw answer to `normalise`, return a `Turn`.
**`normalise` is the part you write**, in exercises 1 to 4. Notice what the client does *not*
contain: a single `if backend == ...`. The differences between backends live in data
and in the four functions you write, never in the agent code that calls the client.

In [ ]:
@dataclass
class ToolCall:
    """One tool call, in the one shape every agent lesson can rely on."""
    id: str
    name: str
    args: dict


@dataclass
class Turn:
    """One model answer, in the one shape every agent lesson can rely on.

    `text` is the visible answer with any reasoning removed. `finish` is one of "stop",
    "tool_call", "length", "filtered", "error" or "other". `data` is the JSON object found in
    `text`, or None. `error` is empty unless the backend rejected the request.
    """
    text: str
    reasoning: str
    tool_calls: tuple
    finish: str
    input_tokens: int | None
    output_tokens: int | None
    data: dict | None
    error: str


WEATHER_TOOL = {"type": "function", "function": {
    "name": "get_weather", "description": "Current weather for a city.",
    "parameters": {"type": "object", "properties": {"city": {"type": "string"}},
                   "required": ["city"]}}}
CITY_SCHEMA = {"type": "object", "properties": {"city": {"type": "string"},
                                                "temp_c": {"type": "number"}},
               "required": ["city", "temp_c"], "additionalProperties": False}
PROMPTS = {
    "plain": "Say hello.",
    "tools": "What is the weather in Lisbon? Use the tool.",
    "reasoning": "Which is larger, 9.9 or 9.11?",
    "schema": "Extract the city and the temperature: 'It is 21 degrees in Lisbon.'",
}


def build_request(probe: str) -> dict:
    """The request for one probe, in the OpenAI chat-completions shape."""
    req = {"messages": [{"role": "user", "content": PROMPTS[probe]}]}
    if probe == "tools":
        req["tools"] = [WEATHER_TOOL]
        req["tool_choice"] = "auto"
    elif probe == "reasoning":
        req["reasoning_effort"] = "low"
    elif probe == "schema":
        req["response_format"] = {"type": "json_schema", "json_schema": {
            "name": "city_temperature", "strict": True, "schema": CITY_SCHEMA}}
    return req


def probe_of(request: dict) -> str:
    """Which probe a request is. The replays cover one feature at a time."""
    used = [name for key, name in (("tools", "tools"), ("reasoning_effort", "reasoning"),
                                   ("response_format", "schema")) if key in request]
    if len(used) > 1:
        raise ValueError("the replays hold single-feature probes only, got " + " + ".join(used))
    return used[0] if used else "plain"


class Replay:
    """Stands in for one backend's endpoint: `post(request)` returns `(status, body)`.

    A live version of this class would POST the request to a base URL with an API key and
    return the same pair. Nothing else in the lesson would change, which is the point.
    """

    def __init__(self, backend: str):
        if backend not in RECORDED:
            raise ValueError(f"unknown backend {backend!r}; choose one of {sorted(RECORDED)}")
        self.backend = backend

    def post(self, request: dict) -> tuple[int, dict]:
        status, body = RECORDED[self.backend][probe_of(request)]
        return status, copy.deepcopy(body)   # a copy, so no caller can corrupt the replay


class Client:
    """The whole client: one wire, one normaliser, one return type."""

    def __init__(self, backend: str):
        self.backend = backend
        self._wire = Replay(backend)

    @classmethod
    def from_env(cls) -> "Client":
        """Pick the backend from one environment variable, defaulting to the local server."""
        return cls(os.environ.get("SYNAPSA_BACKEND", "local"))

    def complete(self, request: dict) -> Turn:
        status, body = self._wire.post(request)
        return normalise(status, body)

## 3. Exercise 1: `parse_tool_calls()`

A tool call is where a client most needs to be strict, because the next thing that happens
is your code *running* something. The chat shape says `function.arguments` is a **string** of
JSON that the model generated, and the specification warns in so many words that the model
does not always generate valid JSON. A client should also tolerate `arguments` arriving as a
real object, as an empty string for a tool that takes nothing, or as `null`, a call with no
`id`, and the older single `function_call` field that the specification marks deprecated.

Return one `ToolCall` per call, in order. A call you cannot read cleanly must raise
`ValueError`; it must never be repaired by guessing and never returned half-parsed.

<details><summary>💡 Hint 1: what to think about</summary>

List every shape `arguments` can take, and for each one decide what the ToolCall's `args`
should be. Then decide which shapes are *errors* rather than odd-but-fine: what is different
about JSON that fails to parse, versus JSON that parses to something that is not an object?
Also: where does a call's default `id` come from when the backend never sent one?
</details>
<details><summary>💡 Hint 2: the approach, in words</summary>

Take the `tool_calls` list if it has anything in it, otherwise the legacy field, otherwise
nothing. For each entry read the name, and handle `arguments` by type: text is parsed with
the standard `json` module and must come out as an object; an object is used as it is;
a missing value and blank text both mean no arguments. Anything else, or a missing name,
raises `ValueError` with the tool's name in the message. Number the calls by their position.
</details>

In [ ]:
def parse_tool_calls(message: dict) -> tuple:
    """Every tool call in a chat-completions `message`, as a tuple of `ToolCall`.

    Reads `message["tool_calls"]`. If that is missing, empty or None, falls back to the single
    legacy `message["function_call"]` (a dict with `name` and `arguments`), which counts as one
    call. Per call:

    * `name` is `function.name`, a non-empty string, else ValueError.
    * `args` comes from `function.arguments`: a string is parsed as JSON and must be a JSON
      object; a dict is used as it is; None, "" and whitespace-only text mean `{}`. Invalid JSON,
      JSON that is not an object, or any other type raises ValueError naming the tool.
    * `id` is the call's `id` if it is a non-empty string, otherwise `f"call_{position}"`,
      where `position` counts from 0 in the list of calls.

    Example:
        >>> m = {"tool_calls": [{"function": {"name": "f", "arguments": "{\\"x\\": 1}"}}]}
        >>> parse_tool_calls(m)
        (ToolCall(id='call_0', name='f', args={'x': 1}),)
        >>> parse_tool_calls({"content": "hello"})
        ()

    Returns: a tuple of ToolCall, empty when the message holds no tool call.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_parse_tool_calls() -> None:
    one = parse_tool_calls({"tool_calls": [{"id": "c1", "function": {
        "name": "get_weather", "arguments": "{\"city\": \"Oslo\"}"}}]})
    assert one == (ToolCall("c1", "get_weather", {"city": "Oslo"}),), (
        f"a standard call came back as {one!r}; the arguments string must be parsed into a dict, "
        "not passed through as text")
    blank = parse_tool_calls({"tool_calls": [{"function": {"name": "ping", "arguments": ""}}]})
    assert blank[0].args == {}, (
        f"a tool that takes no arguments sends an empty string; args should be {{}}, got "
        f"{blank[0].args!r}. json.loads('') raises, so test for blank text before parsing")
    second = parse_tool_calls({"tool_calls": [
        {"id": "a", "function": {"name": "f", "arguments": {"x": 1}}},
        {"function": {"name": "g", "arguments": None}}]})
    assert second[0].args == {"x": 1} and second[1].id == "call_1", (
        f"got {second!r}; an object passes through unchanged, and a call with no id is numbered "
        "by its position in the list (the second call here)")
    pair = parse_tool_calls({"tool_calls": [
        {"id": "w2", "function": {"name": "get_weather", "arguments": "{\"city\": \"Paris\"}"}},
        {"id": "w1", "function": {"name": "get_weather", "arguments": "{\"city\": \"Oslo\"}"}}]})
    assert [(c.id, c.args.get("city")) for c in pair] == [("w2", "Paris"), ("w1", "Oslo")], (
        f"got {pair!r}; two calls to the same tool are two calls (a model may ask for two cities "
        "at once), kept in the order they arrived: do not key them by name or sort them")
    for bad, why in (("{not json", "invalid JSON"), ("[1, 2]", "a JSON array")):
        try:
            parse_tool_calls({"tool_calls": [{"function": {"name": "f", "arguments": bad}}]})
        except ValueError:
            continue
        raise AssertionError(f"arguments {bad!r} ({why}) must raise ValueError, not return a "
                             "call your agent would then run")
    print("exercise 1 looks right: one ToolCall type, however the arguments arrived")

In [ ]:
_try("exercise 1", _check_parse_tool_calls)

## 4. Exercise 2: `split_reasoning()`

There is no single place a reasoning model's thinking goes. The replays and the
documentation give four:

1. `message.reasoning_content`, a string: llama.cpp's server returns reasoning under this
   name, which it borrows from the DeepSeek API.
2. `message.reasoning`, a string: the aggregator's documented field.
3. `message.reasoning_details`, a list of typed blocks: `reasoning.text` blocks carry `text`,
   `reasoning.summary` blocks carry `summary`, and `reasoning.encrypted` blocks carry opaque
   `data` that is not text.
4. `<think>...</think>` tags inside `content`: what a server leaves when told not to parse
   thoughts.

Return `(text, reasoning)`: the visible answer with **every** think block removed, and the
reasoning taken from the first of those four sources that has any. The trap is a server in
the mode that fills field 1 *and* leaves the tags in `content`: the reasoning must appear
once, and the answer must not carry the tags.

<details><summary>💡 Hint 1: what to think about</summary>

Two jobs, and they are independent: cleaning `content`, and choosing where the reasoning
comes from. Which of them must run even when a reasoning field is present? Also think about
a model cut off by its token limit in the middle of thinking: what does `content` look like
then, and what is the answer?
</details>
<details><summary>💡 Hint 2: the approach, in words</summary>

Clean `content` first: cut out every complete tag pair, matching lazily so that two blocks do
not swallow the text between them and letting a dot cross line breaks, then treat an opening
tag with no closing partner as a block that runs to the end. Then walk the four sources in
the order given and stop at the first non-blank one; the inline blocks are the last resort.
Strip whitespace from both results, and join several details blocks with a newline.
</details>

In [ ]:
def split_reasoning(message: dict) -> tuple[str, str]:
    """`(text, reasoning)` for a chat-completions `message`.

    `text` is `message["content"]` (None counts as "") with every `<think>...</think>` block
    removed, then stripped. An opening `<think>` with no closing tag (an answer cut off
    mid-thought) is a block that runs to the end of the content. `reasoning` is the first
    non-blank of: `reasoning_content` (str), `reasoning` (str), the `reasoning_details` blocks
    (the `text` of each `reasoning.text` and the `summary` of each `reasoning.summary`, in list
    order, joined by a newline; `reasoning.encrypted` blocks are skipped), and finally the
    inline think blocks (joined by a newline). Both results are stripped; "" when absent.

    Example:
        >>> split_reasoning({"content": "<think>a</think>Hi", "reasoning": "b"})
        ('Hi', 'b')
        >>> split_reasoning({"content": "Hi <think>never finished"})
        ('Hi', 'never finished')

    Returns: a `(text, reasoning)` pair of strings.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_split_reasoning() -> None:
    field = split_reasoning({"content": "Answer.", "reasoning_content": "because"})
    assert field == ("Answer.", "because"), (
        f"got {field!r}; a reasoning string field is returned as the reasoning, and the content "
        "is the answer")
    inline = split_reasoning({"content": "<think>step one\nstep two</think>\n\nAnswer."})
    assert inline == ("Answer.", "step one\nstep two"), (
        f"got {inline!r}; a think block can span lines (let the dot match a newline), and the "
        "text left over must be stripped")
    both = split_reasoning({"content": "<think>why</think>Yes.", "reasoning_content": "why"})
    assert both == ("Yes.", "why"), (
        f"got {both!r}; when a field AND tags are both present the reasoning appears once and "
        "the tags never reach the visible text, so clean content even when a field wins")
    two = split_reasoning({"content": "<think>a</think>X<think>b</think>Y"})
    assert two == ("XY", "a\nb"), (
        f"got {two!r}; two blocks must be matched lazily, or the greedy match eats the X between")
    blocks = split_reasoning({"content": "ok", "reasoning_details": [
        {"type": "reasoning.text", "text": "one"},
        {"type": "reasoning.encrypted", "data": "AAAA"},
        {"type": "reasoning.summary", "summary": "two"}]})
    assert blocks == ("ok", "one\ntwo"), (
        f"got {blocks!r}; join the text and summary blocks in order and skip the encrypted one, "
        "whose `data` is not reasoning text")
    cut = split_reasoning({"content": "Hi <think>never finished"})
    assert cut == ("Hi", "never finished"), (
        f"got {cut!r}; an opening tag with no closing tag (an answer cut off mid-thought) is a "
        "block that runs to the end, and a loop looking for the closing tag must still stop")
    print("exercise 2 looks right: reasoning found wherever it was put, and kept out of the answer")

In [ ]:
_try("exercise 2", _check_split_reasoning)

## 5. Exercise 3: `recover_json()`

You asked for JSON and the request said `strict`. Did you get JSON? The `schema` replays,
each written to its product's documentation, differ by backend: one returns a bare object, one rejects the request, and one
answers politely in prose with the object inside a fenced code block. A client that only
calls `json.loads` on `content` treats the third as a failure. A client that pulls out
everything between the first `{` and the last `}` is wrong the moment a sentence contains a
brace.

Return the **first JSON object** that appears anywhere in the text, or `None`.

<details><summary>💡 Hint 1: what to think about</summary>

A regular expression cannot count braces, and a string value may itself contain `}`. The
standard library already knows how to decide where a JSON value ends. What does it offer that
parses from the middle of a string instead of from the start, and what does it do when the
text at that position is not JSON at all?
</details>
<details><summary>💡 Hint 2: the approach, in words</summary>

Try every position that holds an opening brace, from the left. At each, ask the decoder to
read one value starting right there. If it fails, move to the next brace. If it succeeds and
the value is an object, stop. No object anywhere means `None`, and so does text that is not a
string.
</details>

In [ ]:
def recover_json(text: str) -> "dict | None":
    """The first JSON object found anywhere in `text`, or None.

    "First" means the object whose opening brace comes earliest among those that begin a
    complete, valid JSON object. Text before it (prose, a code fence) and after it is ignored;
    a brace that does not begin valid JSON is skipped, and so is a brace inside a string value.
    A `text` that is not a str returns None.

    Example:
        >>> recover_json('Sure! {"a": 1} Anything else?')
        {'a': 1}
        >>> recover_json('a {curly} word, then {"b": {"c": "}"}}')
        {'b': {'c': '}'}}
        >>> recover_json("no object here") is None
        True

    Returns: the parsed dict, or None.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_recover_json() -> None:
    assert recover_json('{"a": 1}') == {"a": 1}, "a bare object must come back as a dict"
    fenced = recover_json('Here you go:\n```json\n{"city": "Oslo", "n": 2}\n```\nDone.')
    assert fenced == {"city": "Oslo", "n": 2}, (
        f"got {fenced!r}; the object inside a fenced block is still the answer, so text on both "
        "sides of it must be ignored")
    tricky = recover_json('Use {braces} sparingly: {"a": "}", "b": {"c": 1}}')
    assert tricky == {"a": "}", "b": {"c": 1}}, (
        f"got {tricky!r}; a `}}` inside a string value must not end the object, and a brace that "
        "does not begin valid JSON (`{braces}`) must be skipped, not fatal")
    assert recover_json('{"a": 1} then {"b": 2}') == {"a": 1}, "the FIRST object wins"
    assert recover_json("no json here") is None and recover_json("") is None, (
        "text with no object returns None, it does not raise")
    assert recover_json('Cut off: {"city": "Os') is None, (
        "an object cut off part-way is not JSON, so the answer is None; make sure the search "
        "still moves past a brace that never closes")
    print("exercise 3 looks right: JSON recovered from prose, fences and awkward braces")

In [ ]:
_try("exercise 3", _check_recover_json)

## 6. Exercise 4: `normalise()`

Now compose. `normalise(status, body)` is the seam between the outside world and every agent
lesson that follows: it takes what a backend sends and returns a `Turn`. Its own new
work is the plumbing around your three functions:

- **Errors.** Three different error bodies exist: `{"error": {"message": ...}}`,
  `{"error": "some text"}`, and none at all. All must become `Turn.error`.
- **Finish reasons.** Backends say `stop`, `tool_calls`, `function_call`, `length`,
  `content_filter`, or nothing. The Turn has one vocabulary.
- **Token counts.** Chat completions call them `prompt_tokens` and `completion_tokens`; the
  same vendor's newer Responses API calls them `input_tokens` and `output_tokens`; a server
  may leave them out. Missing is `None`, never `0`, because zero would be a claim.

<details><summary>💡 Hint 1: what to think about</summary>

What does an agent loop do with a Turn whose `finish` is `"stop"` but which carries tool
calls? Decide which fact should win. Also decide what a malformed tool call should do to the
whole turn: raise into the caller, or become a Turn the loop can inspect?
</details>
<details><summary>💡 Hint 2: the approach, in words</summary>

Deal with a failing status first and return early. Then take the first choice's message and
run your three functions on it; if tool-call parsing raises, return an error Turn instead of
letting the exception out. Translate the finish reason with a small lookup, defaulting to the
"other" bucket, and let the presence of tool calls decide when the backend said stop. Read
token counts by trying the two names in turn and accepting only real integers.
</details>

In [ ]:
_FINISH = {"stop": "stop", "tool_calls": "tool_call", "function_call": "tool_call",
           "length": "length", "content_filter": "filtered"}


def normalise(status: int, body: dict) -> Turn:
    """Turn one raw `(status, body)` from any backend into a `Turn`.

    * `status >= 400`: return a Turn with `finish="error"`, empty text, no calls, and `error`
      set to `body["error"]["message"]` if `error` is a dict with a non-empty string `message`,
      else `body["error"]` if that is a non-empty string, else `f"HTTP {status}"`.
    * a body with no choices: the same error Turn, with `error="response has no choices"`.
    * otherwise read `body["choices"][0]`: `text` and `reasoning` from `split_reasoning`,
      `tool_calls` from `parse_tool_calls`, `data` from `recover_json(text)`. If
      `parse_tool_calls` raises ValueError, return an error Turn whose `error` starts with
      `"bad tool call: "` followed by the message; the exception must not escape.
    * `finish` is `finish_reason` mapped through the lookup above (`stop`, `tool_call`,
      `length`, `filtered`), anything else being "other"; but a turn that carries tool calls and
      would be "stop" or "other" is "tool_call", because the calls are what the caller must act
      on. A "length" turn stays "length".
    * `input_tokens` is `usage["prompt_tokens"]`, else `usage["input_tokens"]`; `output_tokens`
      is `usage["completion_tokens"]`, else `usage["output_tokens"]`. A value that is not an int
      (a bool is not an int here), or a missing `usage`, gives None.

    Example:
        >>> t = normalise(200, {"choices": [{"finish_reason": "stop",
        ...                                  "message": {"content": "Hi"}}]})
        >>> (t.text, t.finish, t.input_tokens)
        ('Hi', 'stop', None)
        >>> normalise(429, {"error": "slow down"}).error
        'slow down'

    Returns: a Turn.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_normalise() -> None:
    def wrap(message, finish="stop", usage=None):
        body = {"choices": [{"finish_reason": finish, "message": message}]}
        if usage is not None:
            body["usage"] = usage
        return body

    turn = normalise(200, wrap({"content": "Hi"}, usage={"prompt_tokens": 3, "completion_tokens": 4}))
    assert (turn.text, turn.finish, turn.input_tokens, turn.output_tokens) == ("Hi", "stop", 3, 4), (
        f"a plain answer came back as {turn!r}")
    alias = normalise(200, wrap({"content": "Hi"}, usage={"input_tokens": 5, "output_tokens": 6}))
    assert (alias.input_tokens, alias.output_tokens) == (5, 6), (
        f"got {(alias.input_tokens, alias.output_tokens)!r}; the input_tokens/output_tokens spelling "
        "must be read when the prompt_tokens spelling is absent")
    assert normalise(200, wrap({"content": "Hi"})).input_tokens is None, (
        "no usage block means None, never 0: zero would claim the call was free")
    calls = normalise(200, wrap({"content": None, "tool_calls": [
        {"function": {"name": "f", "arguments": "{}"}}]}))
    assert calls.finish == "tool_call" and len(calls.tool_calls) == 1, (
        f"got finish={calls.finish!r}; a turn that carries tool calls but says 'stop' is a "
        "'tool_call' turn, because the calls are what the caller must act on")
    assert normalise(200, wrap({"content": "cut"}, finish="length")).finish == "length", (
        "'length' must survive normalisation")
    for code in (400, 429, 500):
        err = normalise(code, {"error": {"message": "nope"}})
        assert err.finish == "error" and err.error == "nope", (
            f"status {code} came back as {err!r}; every status of 400 and above is an error Turn")
    assert normalise(503, {}).error == "HTTP 503", "no error body still needs a non-empty reason"
    bad = normalise(200, wrap({"tool_calls": [{"function": {"name": "f", "arguments": "{oops"}}]}))
    assert bad.finish == "error" and bad.error.startswith("bad tool call"), (
        f"got {bad!r}; a malformed tool call becomes an error Turn, it must not raise out of "
        "the client")
    print("exercise 4 looks right: any backend's answer arrives as one Turn")

In [ ]:
_try("exercise 4", _check_normalise)

**Same call, three formats, one type.** With `normalise` working you can now put the
replays from section 1 through the client and count what survives. The numbers below are
computed from the replays, not typed.

In [ ]:
def _same_call_demo() -> None:
    raw_strings, calls = [], []
    for backend in BACKENDS:
        raw = RECORDED[backend]["tools"][1]["choices"][0]["message"]["tool_calls"][0]
        raw_strings.append(raw["function"]["arguments"])
        turn = Client(backend).complete(build_request("tools"))
        (call,) = turn.tool_calls
        calls.append((call.name, json.dumps(call.args, sort_keys=True)))
        print(f"{backend:<9} finish={turn.finish:<9} {call.name}({call.args})  id={call.id}")
    print(f"\ndistinct raw `arguments` strings: {len(set(raw_strings))}"
          f"   distinct normalised calls: {len(set(calls))}")
    assert len(set(calls)) == 1, "the three backends asked for different things after normalising"


_try("same call", _same_call_demo, needs=("exercise 1", "exercise 2", "exercise 3", "exercise 4"))

## 7. Exercise 5: `classify()`, the compatibility suite's judge

`normalise` hides differences; the suite exists to *report* them. For each replayed
`(status, body)` and the feature that was requested, `classify` returns exactly one verdict.
The reference is the shape this lesson's own client is written against: the OpenAI chat
shape, plus `reasoning_content` for reasoning, which that shape does not define. The choice
of reference is a convention, and the table states it:

| feature | **kept** | **renamed** | **dropped** |
|---|---|---|---|
| `plain` | `content` has text | (never) | `content` is blank |
| `tools` | `tool_calls` present and every `arguments` a string | tool calls present in another shape (`arguments` not a string) or only a legacy `function_call` | no call at all |
| `reasoning` | non-blank `reasoning_content` | reasoning in `reasoning`, in a non-empty `reasoning_details` list, or in `<think>` tags | none anywhere |
| `schema` | `content` is, after stripping, a JSON object | an object recoverable from the text | no object |

Any status of 400 or above is **rejected**, for every feature. **HTTP 200 is not evidence of
support**: one of the stand-ins accepts the `schema` request, answers 200, and ignores the
constraint, as its vendor's documentation describes. Only the body can tell you.

<details><summary>💡 Hint 1: what to think about</summary>

Do the checks in an order where each one can only reach the next if the previous did not
decide. What does `rejected` need to look at? What does `kept` for `schema` need that
`renamed` does not? And should a reasoning field that is present but blank count as present?
</details>
<details><summary>💡 Hint 2: the approach, in words</summary>

Check the feature name and the status first. Then read `content` defensively (it may be
None). For each feature, test the strict form first and return kept; then the tolerated forms
and return renamed; then fall through to dropped. For `schema`, the strict form is `json.loads`
on the stripped content producing a dict; for the tolerated form reuse your `recover_json`.
</details>

In [ ]:
def classify(feature: str, status: int, body: dict) -> str:
    """One of "kept", "renamed", "dropped" or "rejected" for one `(status, body)` answer.

    `feature` is one of `PROBES`, else ValueError (checked first). A `status` of 400 or above
    is "rejected" whatever the body says. Otherwise read `body["choices"][0]["message"]` and
    apply the table above. "Blank" means empty or whitespace only. For "reasoning", a
    non-empty `reasoning_details` list counts whatever its blocks hold (encrypted reasoning is
    still reasoning, in another form), and so does an opening `<think>` with no closing tag.
    For "schema", "kept" means `json.loads(content.strip())` gives a dict; "renamed" means
    `recover_json(content)` finds one; text with no object in it, such as a bare array of
    numbers or a scalar, is "dropped".

    Example:
        >>> ok = {"choices": [{"message": {"content": '{"a": 1}'}}]}
        >>> classify("schema", 200, ok)
        'kept'
        >>> classify("schema", 400, {"error": "no"})
        'rejected'

    Returns: the verdict string.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_classify() -> None:
    def msg(**fields):
        return {"choices": [{"message": fields}]}

    assert classify("plain", 200, msg(content="hi")) == "kept", "plain text is 'kept'"
    for feature in PROBES:
        assert classify(feature, 400, {"error": "x"}) == "rejected", (
            f"{feature}: status 400 is 'rejected' before the body is even read; a rejection has no "
            "`choices` to look at")
    std = msg(tool_calls=[{"function": {"name": "f", "arguments": "{}"}}])
    obj = msg(tool_calls=[{"function": {"name": "f", "arguments": {}}}])
    assert (classify("tools", 200, std), classify("tools", 200, obj),
            classify("tools", 200, msg(content="I cannot"))) == ("kept", "renamed", "dropped"), (
        "tools: arguments as a string is the documented shape (kept); arguments as an object is "
        "the same call in another shape (renamed); no call at all is dropped")
    got = (classify("reasoning", 200, msg(content="a", reasoning_content="r")),
           classify("reasoning", 200, msg(content="a", reasoning="r")),
           classify("reasoning", 200, msg(content="<think>r</think>a")),
           classify("reasoning", 200, msg(content="a")))
    assert got == ("kept", "renamed", "renamed", "dropped"), (
        f"reasoning verdicts came back {got!r}; `reasoning_content` is the reference name, any "
        "other place is 'renamed', and none is 'dropped'")
    fenced = msg(content='```json\n{"a": 1}\n```')
    assert (classify("schema", 200, msg(content=' {"a": 1} ')), classify("schema", 200, fenced),
            classify("schema", 200, msg(content="sorry"))) == ("kept", "renamed", "dropped"), (
        "schema: bare JSON (surrounding whitespace allowed) is kept, JSON wrapped in a fence is "
        "renamed, no JSON is dropped. A 200 does not mean the request field was honoured")
    print("exercise 5 looks right: a verdict for every cell, from the body and never from the status alone")

In [ ]:
_try("exercise 5", _check_classify)

## 8. The compatibility suite

Given: `run_suite` sends every probe to every backend and asks `classify` for the verdict.
Run it. This table is the deliverable of the lesson, and it is computed from the hand-written
replays above, so it is only as fresh as the snapshot date printed with it, and it shows the
method rather than a measurement of any provider.

In [ ]:
def run_suite() -> dict[tuple[str, str], str]:
    """`{(backend, probe): verdict}` for every backend and probe."""
    verdicts = {}
    for backend in BACKENDS:
        wire = Replay(backend)
        for probe in PROBES:
            status, body = wire.post(build_request(probe))
            verdicts[(backend, probe)] = classify(probe, status, body)
    return verdicts


def _show_matrix() -> None:
    verdicts = run_suite()
    print(f"compatibility matrix, hand-written replays as of {SNAPSHOT_DATE}\n")
    print(f"{'':<10}" + "".join(f"{probe:<11}" for probe in PROBES) + "kept of total")
    for backend in BACKENDS:
        row = [verdicts[(backend, probe)] for probe in PROBES]
        print(f"{backend:<10}" + "".join(f"{v:<11}" for v in row)
              + f"{row.count('kept')} of {len(row)}")
    tally = {v: sum(1 for x in verdicts.values() if x == v)
             for v in ("kept", "renamed", "dropped", "rejected")}
    print("\ncells by verdict: " + ", ".join(f"{k} {v}" for k, v in tally.items()))


_try("compatibility matrix", _show_matrix, needs=("exercise 5",))

Read the table by verdict. A **`renamed`** cell costs you little once `normalise` exists: the
information arrived, in another place. For `schema` it costs one more step, because an object
fished out of prose was never held to the schema, so validate it before you use it. A
**`dropped`** cell costs you the feature: nothing about the request was wrong and no error was
raised, yet the feature is missing from the answer, and the documentation of the product that
stand-in imitates says the field is ignored. A **`rejected`** cell at least fails loudly. The
next cell reads the fields straight out of the replays instead of judging them, so you can
see which keys each backend adds to the reference message.

In [ ]:
REFERENCE_MESSAGE_FIELDS = {"role", "content", "tool_calls"}


def _field_inventory() -> None:
    print(f"message fields beyond the reference {sorted(REFERENCE_MESSAGE_FIELDS)}, in the replays as of {SNAPSHOT_DATE}")
    for backend in BACKENDS:
        seen = set()
        for probe in PROBES:
            status, body = RECORDED[backend][probe]
            if status < 400:
                seen |= set(body["choices"][0]["message"])
        extras = sorted(seen - REFERENCE_MESSAGE_FIELDS)
        print(f"  {backend:<9} {extras if extras else 'none'}")
    print("\nA field a backend adds is harmless until your code reads it by name: that is how a")
    print("client written against one backend fails quietly on the next.")


_field_inventory()

## 9. Switching backends with one environment variable

The payoff. The function below is the agent code: it asks for the weather and reads the
call. It has no idea which backend it is talking to. The cell switches `SYNAPSA_BACKEND`
between runs and nothing else changes.

In [ ]:
def weather_agent_step(client: Client) -> str:
    """The agent code: ask, then read the one tool call. Knows nothing about backends."""
    turn = client.complete(build_request("tools"))
    if turn.finish != "tool_call":
        return f"no tool call (finish={turn.finish}, error={turn.error!r})"
    call = turn.tool_calls[0]
    return f"{call.name}({json.dumps(call.args, sort_keys=True)})"


def _env_switch_demo() -> None:
    saved = os.environ.get("SYNAPSA_BACKEND")
    try:
        for backend in BACKENDS:
            os.environ["SYNAPSA_BACKEND"] = backend
            print(f"SYNAPSA_BACKEND={backend:<9} -> {weather_agent_step(Client.from_env())}")
    finally:
        if saved is None:
            os.environ.pop("SYNAPSA_BACKEND", None)
        else:
            os.environ["SYNAPSA_BACKEND"] = saved


_try("environment switch", _env_switch_demo,
     needs=("exercise 1", "exercise 2", "exercise 3", "exercise 4"))

## 10. Common mistakes

- **Believing "OpenAI-compatible" means "behaves the same".** It means the request is
  accepted. The frontier stand-in accepts `response_format` and `reasoning_effort` and
  ignores both, because it is written to match its vendor's documentation, which says both
  are ignored. The matrix above shows what checking compatibility looks like; only real
  traffic can tell you what a real provider does today.
- **Trusting the status code.** A 200 with a prose answer to a schema request means the schema
  was not enforced, whatever the status says. Check the body, as `classify` does, and validate
  any object you recover.
- **Calling `json.loads` on tool arguments and running the result.** The specification says
  in so many words that a model does not always produce valid JSON and can invent parameters.
  Parse, then validate against the schema, then run.
- **Reading reasoning from one field name.** This lesson's client handles four places, and
  Ollama's own API adds another, `message.thinking`. Your client should gain a fifth
  spelling by editing one function.
- **Branching on the backend name in agent code.** Every `if backend == ...` is a difference
  that should have been a fact in the adapter.
- **Re-using a recording after its snapshot date.** Provider field names change. The lesson
  prints the date so that stale facts look stale.

One of those mistakes is worth checking rather than believing. This cell prints the `schema`
probe as each stand-in replays it.

In [ ]:
def _status_is_not_support() -> None:
    ok, kept = 0, 0
    print("the `schema` probe, as each stand-in replays it:")
    for backend in BACKENDS:
        status, body = RECORDED[backend]["schema"]
        verdict = classify("schema", status, body)
        ok += status == 200
        kept += verdict == "kept"
        print(f"  {backend:<9} HTTP {status}  verdict {verdict}")
    only = "only " if kept < ok else ""
    print(f"\n{ok} of {len(BACKENDS)} stand-ins replay HTTP 200, and {only}{kept} of those "
          f"carr{'ies' if kept == 1 else 'y'} the object bare. An HTTP 200 tells you the request "
          "was accepted, never that it was honoured.")


_try("status is not support", _status_is_not_support, needs=("exercise 5",))

## 11. Self-check

1. A backend answers HTTP 200 to a request carrying a JSON schema, and the answer is a
   sentence with a JSON object inside a code fence. In this lesson's terms:
   - (a) the feature was honoured, since there was no error
   - (b) the request was accepted, but the constraint was not enforced, so the client must
         recover the object and validate it itself
   - (c) the request was rejected

2. You add a fourth backend that returns reasoning in a field called `thinking`. The change
   belongs in:
   - (a) every agent that uses the client
   - (b) the model
   - (c) the one function that finds the reasoning, so nothing above the client notices

3. Tool arguments arrive as a string of JSON, and you have parsed it into a dict. Before
   running the tool you should still:
   - (a) validate the dict against the tool's schema, because a model can invent parameters
   - (b) do nothing more: a dict that parsed is correct
   - (c) parse it a second time

4. A reasoning model is cut off by its token limit part-way through its thinking, so
   `content` holds an opening think tag and no closing one. The right visible answer is:
   - (a) the whole content, tag included
   - (b) an error: the response is unusable
   - (c) whatever came before the tag, and the unfinished thought is reasoning

5. In a few months you re-record the four probes against a live provider, run the suite
   again, and one cell of its matrix differs from the one this notebook printed. The most
   likely reason is:
   - (a) a bug in `classify`
   - (b) the provider changed a field name or a behaviour since the snapshot date
   - (c) Python changed how dictionaries print

Answers, with the reasoning, come with this lesson's worked solution when you enrol on Synapsa.

## What you built, and where it goes next

You built the seam every later agent and evaluation lesson stands on: one `Turn` type, one
client, and a suite that says, per backend and per feature, what survives the trip. To make it
live you would replace `Replay.post` with a function that POSTs the request to a base URL with
an API key and returns the same `(status, body)` pair, then re-record the four probes per
backend and update `SNAPSHOT_DATE`. Nothing above that line changes. Run the suite each
quarter: it is cheap, and provider field names change more often than any algorithm does.

In [ ]:
_MARKS = {"passed": "✅", "failed": "❌", "not started": "⏳"}


def _progress_board() -> None:
    """One line per exercise, from the latest run of its check, then the tally."""
    width = max(len(", ".join(funcs)) for funcs in _EXERCISES.values())
    print("progress board")
    for label, funcs in _EXERCISES.items():
        state = _STATUS.get(label, "not started")
        print(f"  {_MARKS[state]} {label:<12} {', '.join(funcs):<{width}}  {state}")
    done = sum(_STATUS.get(label) == "passed" for label in _EXERCISES)
    print(f"\n{done} of {len(_EXERCISES)} exercises complete")
    failing = [label for label in _EXERCISES if _STATUS.get(label) == "failed"]
    if failing:
        print("failing right now: " + ", ".join(failing) + ". Each one printed what went "
              "wrong in its own cell above, and every exercise has hints you can open.")
    elif done < len(_EXERCISES):
        print("work top to bottom: every exercise has hints you can open above its code.")


# Your progress board. Every check is re-run here, quietly, against your code as it stands
# now (each one already printed its feedback in its own cell above), so the board is
# current even if you edited an exercise and did not re-run its check.
if __name__ == "__main__":
    with contextlib.redirect_stdout(io.StringIO()):
        for _name, _check in (("exercise 1", _check_parse_tool_calls),
                              ("exercise 2", _check_split_reasoning),
                              ("exercise 3", _check_recover_json),
                              ("exercise 4", _check_normalise),
                              ("exercise 5", _check_classify)):
            _try(_name, _check)
    _progress_board()
    # A stub you have not reached yet is not a failure. A check that ran and came back wrong
    # is: in a script or under CI it ends this run non-zero, rather than letting a green exit
    # code paper over it. Inside a notebook kernel the board above has already said so, in a
    # line rather than a traceback at the foot of the page.
    if _FAILED_CHECKS and "ipykernel" not in sys.modules:
        raise SystemExit("checks failed: " + ", ".join(dict.fromkeys(_FAILED_CHECKS)))

<!-- COMMONS NOTICE v1 · generated by tools/notebooks.py · do not edit by hand -->
---
**Synapsa Commons** · © 2026 RealAI · licensed under [CC BY-NC-SA
4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/)

**You may** use this lesson to learn and to teach, and copy, fork, share and adapt it.

**You must** credit "Synapsa Commons by RealAI" with a link to
https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials,
say what you changed, and share anything you adapt under this same licence.

**You may not** use it, or anything adapted from it, in a way primarily intended for
commercial advantage or payment: for example selling it, charging for a course, bootcamp or
training built on it, or packaging it into a paid product or service. For a commercial
licence, contact [RealAI](https://www.realai.eu/contact).

Third-party material in this lesson keeps its own licence, named in `assets/SOURCE.md` or
`claims.yaml`. The Synapsa name and logo belong to RealAI and are not licensed. This summary
is not the licence: the [legal
code](https://creativecommons.org/licenses/by-nc-sa/4.0/legalcode) governs.